In [3]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

# Model
model = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
MAX_TOKENS = 150

In [6]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [7]:

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [8]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [9]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Nitish."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 10
Hi, my name is Nitish.


'Hello Nitish! 👋 Nice to meet you. How can I assist you today?'

In [10]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 44
Hi, my name is Nitish.
Hello Nitish! 👋 Nice to meet you. How can I assist you today?
Can you explain short term memory?


'### Short‑Term Memory (STM) – A Quick Overview\n\n| Aspect | What It Means | Typical Numbers |\n|--------|----------------|-----------------|\n| **Definition** | The temporary storage system that holds a limited amount of information for a brief period (seconds to a few minutes) while we are actively using it. | — |\n| **Capacity** | Often cited as **7\u202f±\u202f2** “chunks” of information (Miller, 1956). Modern research suggests the true capacity may be closer to **4‑5** chunks, especially when the material is unfamiliar. | 4‑7 items (or “chunks”) |\n| **Duration** | Without rehearsal, information fades in **≈\u202f15‑30\u202fseconds**. With active rehearsal (repeating or rehearsing the material), it can be maintained much longer. | 15‑30\u202fs (no rehearsal) |\n| **Encoding** | Primarily **acoustic** (sound) for verbal material, though visual and spatial codes are also used (e.g., remembering a phone number as a visual pattern). | — |\n| **Retrieval** | Items are accessed directl

In [11]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


'I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [12]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Nitish.
------------------------------------------------------------------------------------------------------------------------
Hello Nitish! 👋 Nice to meet you. How can I assist you today?
------------------------------------------------------------------------------------------------------------------------
Can you explain short term memory?
------------------------------------------------------------------------------------------------------------------------
### Short‑Term Memory (STM) – A Quick Overview

| Aspect | What It Means | Typical Numbers |
|--------|----------------|-----------------|
| **Definition** | The temporary storage system that holds a limited amount of information for a brief period (seconds to a few minutes) while we are actively using it. | — |
| **Capacity** | Often cited as **7 ± 2** “chunks” of information (Miller, 1956). Modern research suggests the true capacity may be closer to **4‑5** chunks, especially when the material is unfamiliar. |